## Python - Debugging et Logging

---

### Détecter et corriger nos erreurs

- Vous avez rencontré des erreurs dans vos programmes :

  - **erreurs de syntaxe** : celles-ci sont généralement détectées avant l'exécution grâce à l'assistance de votre environnement de développement

  - **exceptions** : celles-ci se produisent pendant l'exécution du programme ; parfois, elles ne sont pas de votre fait et on a vu comment les récupérer, mais d'autres fois elles sont dues à des erreurs dans votre code

  - **erreurs logiques** : ne provoquent pas d'exceptions, mais entraînent des résultats et des comportements incorrects du programme

- Nous allons voir ici quelques techniques de **vérification** et de ***debugging*** pour vous aider à corriger ces deux derniers types d'erreurs

- On apprendra également l'importance de la **journalisation** (logging), notamment pour les scripts qui s'exécutent de manière autonome sur une infrastructure

---

### Assertions

- Une **assertion** est une instruction qui **vérifie qu'une condition est vraie** ; si ce n'est pas le cas, une exception `AssertionError` est automatiquement levée (déclenchée)

- Et donc, en quoi ça diffère d'une instruction `if` ?

  - on utilise une instruction `if` pour gérer les traitements conditionnels dans le traitement « normal » du programme : pour définir les comportements souhaités en fonction de ceci ou cela

  - Une **assertion** est utilisée pour **vérifier des hypothèses** que le programmeur fait sur le code : par exemple, qu'un nombre doit être positif ici, ou qu'une liste ne doit pas être vide là

  - **Si l'hypothèse est fausse**, cela indiquera un **bug** dans le code

  - C'est pourquoi on utilise les assertions : on veut vérifier dès que possible que le code est sain, et si ce n'est pas le cas, on veut le savoir rapidement avec une erreur claire

- On pourrait traduire ça par : « _je m'assure ici que cette condition est vraie, sinon ça veut dire qu'il y a un bug quelque part, donc il faut immédiatement stopper le programme_ »

- Une assertion est composée :

  - du mot-clé `assert`

  - d'une condition à vérifier (booléen, comme pour un `if`)

  - une virgule

  - un message optionnel à afficher si l'assertion échoue

- Voici un exemple d'assertion qui vérifie au début d'une fonction que l'argument reçu est bien un entier, et qu'en plus il est positif :

In [4]:
def process_number(nb):
  # La fonction built-in 'isinstance' renverra ici True si 'nb' est un entier
  assert isinstance(nb, int), "L'argument doit être un entier"
  assert nb > 0, "L'argument doit être un entier positif"
  # Traitement de l'entier positif
  print(nb)

process_number(10)   # OK car 10 est un entier positif
process_number(2.5)  # fera échouer le 1er assert
process_number(-12)  # fera échouer le 2ème assert (commenter la ligne du dessus pour y arriver)

10


AssertionError: L'argument doit être un entier

- Les exceptions déclenchées par les `assert()` sont les seules que vous ne devez pas gérer avec des blocs `try`/`except` : si une assertion échoue, le programme **doit** s'arrêter, ce n'est pas une situation d'erreur « normale » qui doit être gérée pendant l'exécution

- C'est pourquoi les assertions ne devraient échouer que pendant la phase de développement et de test du script : une fois le code déployé en production, les assertions ne devraient plus jamais échouer

---

### Debugging avec `print()`

- Le « _debugging_ » (débogage) est le processus qui consiste à **localiser et corriger les erreurs** d'exécution dans un programme (erreurs logiques ou exceptions)

- Une technique de _debugging_ très simple est d'insérer des instructions `print()` à différents endroits stratégiques du code, sur le chemin qui mène à l'endroit où l'erreur se produit

  - en vérifiant ainsi visuellement l'évolution des valeurs des variables, on peut repérer où les choses tournent mal

  - pour le cas précis de traitements de bugs, ces affichages viennent en complément des assertions, qui permettent de voir qu'il y a un problème, mais pas ce qui _comment_ on a abouti à ce problème

- Voici un petit programme orienté réseau qui télécharge le contenu d'une page web ; on y a inséré des `print()` pour suivre l'évolution des variables :

In [5]:
import requests

def fetch_page(url):
  print(f"URL : {url}")
  response = requests.get(url)
  print(f"Code de statut réponse : {response.status_code}")
  content_length = len(response.text)
  print(f"Content length : {content_length}")
  return response.text

page_content = fetch_page('https://motherfuckingwebsite.com')
print(f"Extrait du contenu : {page_content[:100]}")

URL : https://motherfuckingwebsite.com
Code de statut réponse : 200
Content length : 4903
Extrait du contenu : <!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=de


- On utilise ici une fonctionnalité nouvelle de `print()` : les ***f-strings***

  - en mettant un `f` avant les guillemets, on peut insérer des expressions Python entre accolades `{}` dans la chaîne de caractères, et celles-ci seront évaluées et remplacées par leur valeur lors de l'affichage

  - c'est plus lisible et souvent plus pratique que de faire des concaténations avec des `+` ou des virgules

- Dans le développement et le test normal d'un script, ces affichages ne sont pas présents à la base car il ne font pas partie du fonctionnement normal du programme (sauf bien sûr s'il est nécessaire que le script soit particulièrement verbeux)

  - on va commencer à mettre ce type d'affichages uniquement lorsqu'un problème est détecté, pour essayer de comprendre ce qui se passe

- Ici on imagine qu'il y a un problème dans la récupération de la page web :

  - on vérifie donc d'abord que l'URL est correcte
  
  - puis on vérifie le code de statut de la réponse HTTP
  
  - et enfin on affiche un extrait du contenu de la page (`page_content[:100]` affiche les 100 premiers caractères de la page)

- Cela nous permet de repérer où ça coince : est-ce que l'URL reçue par paramètre est mal formée ? Est-ce que le serveur répond bien, ou bien on a du 400, du 500 ? Est-ce que le contenu de la page ressemble à ce qui est attendu ?

- Apprendre à placer ces affichages de manière judicieuse est une compétence : il faut savoir _quelles informations_ sont pertinentes pour comprendre le problème, et _où les afficher_ pour qu'elles soient utiles

#### Debugging avec `print()` - Limitations

- Il y a un problème avec cette technique : une fois le bug corrigé, il faut **penser à retirer tous les `print()`** que l'on a ajoutés pour le debugging

  - souvent, les concepteurs de scripts se contentent de les commenter (« on ne sait jamais, on pourrait encore en avoir besoin »)

  - parfois, ils oublient certains affichages, qui restent dans le script utilisé sur l'infra et polluent les consoles d'affichage ; ou bien l'autre option : ils enlèvent certains affichages qu'ils croient être pour le debugging et qui sont en fait nécessaires à l'utilisateur
  
  - ou encore : ils ne prennent pas la peine de les retirer du tout...

- De plus on souhaiterait pouvoir activer/désactiver ces affichages de debugging sans avoir à modifier le code source du script : idéalement, on aimerait pouvoir le faire via une option de ligne de commande, ou un paramètre de configuration ; on aimerait peut-être aussi _sauvegarder_ certaines informations pour analyse

- Donc on peut conserver cette solution simple pour le debugging rapide pendant le développement, ou pour les petits scripts utilitaires type _one-shot_, mais pour les scripts plus sérieux, qui seront utilisés sur le long terme et maintenus, on choisira une solution plus robuste : la **journalisation**

---

### Journalisation (Logging)

- La **journalisation** (ou _logging_ en anglais) est une technique qui consiste à enregistrer des messages d'information sur l'exécution d'un programme, **qui ne concernent pas l'utilisateur**, mais qui sont à destination des concepteurs de scripts, administrateurs systèmes/réseaux, SOC (centre opérationnel de sécurité), systèmes de surveillance/diagnostique...

- On peut loguer sur différents canaux en fonction des besoins : la console standard ou d'erreur, des fichiers de log, des systèmes de gestion de logs centralisés, des bases de données...

#### Le module `logging`

- La bibliothèque standard `logging` offre des fonctionnalités avancées pour gérer la journalisation : gestion des niveaux de gravité, redirection des messages vers différents canaux, possibilité de formater les messages

- Reprenons le script précédent en utilisant ce module :

In [9]:
import requests, logging

logging.basicConfig(level=logging.DEBUG, format=' %(asctime)s -  %(levelname)s -  %(message)s')

def fetch_page(url):
  logging.debug('Début fonction fetch_page() - URL soumise : ' + url)
  response = requests.get(url)
  logging.debug('Réponse HTTP - Code de statut : ' + str(response.status_code))
  content_length = len(response.text)
  logging.debug('Content length : ' + str(content_length))
  logging.debug('Fin fonction fetch_page()')
  return response.text

page_content = fetch_page('https://motherfuckingwebsite.com')
logging.debug('Extrait du contenu de la page récupérée : \n' + page_content[:100])

 2025-10-28 09:12:33,486 -  DEBUG -  Début fonction fetch_page() - URL soumise : https://motherfuckingwebsite.com
 2025-10-28 09:12:33,488 -  DEBUG -  Starting new HTTPS connection (1): motherfuckingwebsite.com:443
 2025-10-28 09:12:34,247 -  DEBUG -  https://motherfuckingwebsite.com:443 "GET / HTTP/1.1" 200 2196
 2025-10-28 09:12:34,248 -  DEBUG -  Réponse HTTP - Code de statut : 200
 2025-10-28 09:12:34,249 -  DEBUG -  Content length : 4903
 2025-10-28 09:12:34,249 -  DEBUG -  Fin fonction fetch_page()
 2025-10-28 09:12:34,251 -  DEBUG -  Extrait du contenu de la page récupérée : 
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=de


- La fonction `logging.basicConfig()` permet de configurer le système de journalisation pour le script :

  - ici on définit le niveau minimal des messages à afficher (`DEBUG` - voir plus bas pour les différents niveaux possibles)

  - et le format des messages, qui inclut ici : horodatage, niveau de gravité, et message de log

- On utilise ensuite les différentes fonctions du module `logging` pour enregistrer des messages : ici on a remplacé les `print()` par des appels à `logging.debug()`, qui enregistre des messages de niveau `DEBUG`

  - on donne en argument le message à loguer

- On voit que les entrées loguées sont plus riches que les simples messages : elles incluent les informations configurées auparavant

#### Log dans fichiers

- Le logging directement sur la console est utile pour le debugging pendant le développement

- Pour les scripts en production, on préfère souvent loguer dans des fichiers : cela permet de conserver un historique des événements de manière simple et pratique, tout en permettant d'analyser ultérieurement

- On ajoute le paramètre nommé `filename` à l'appel de `logging.basicConfig()` pour spécifier le fichier de log :

```python
logging.basicConfig(level=logging.DEBUG,
                    format=' %(asctime)s -  %(levelname)s -  %(message)s',
                    filename='mon_script.log')
```

#### Niveaux de log

- Les **niveaux de log** permettent de catégoriser les messages en fonction de leur gravité/importance

  - cela permet de filtrer les messages affichés/enregistrés en fonction du contexte : par exemple, en développement on peut vouloir tout voir (niveau `DEBUG`), tandis qu'en production on peut vouloir ne voir que les erreurs (niveau `ERROR` ou `CRITICAL`)

- Il y a cinq niveaux (et c'est un standard dans les systèmes de logging) :

  | Niveau | Fonction | Description |
  |--------|----------|-------------|
  | `DEBUG` | `logging.debug()` | _(niveau le plus bas)_ informations très détaillées, normalement utiles seulement pour le debugging |
  | `INFO` | `logging.info()` | Informations sur événements généraux, confirmation que quelque chose a bien été fait... ; « jalons » posés sur l'état normal du programme |
  | `WARNING` | `logging.warning()` | Indication d'un problème potentiel / situation inhabituelle, mais n'empêche pas le programme de fonctionner _pour l'instant_ |
  | `ERROR` | `logging.error()` | Une erreur s'est produite, le script a _échoué_ à faire quelque chose, empêche une partie du programme de fonctionner correctement |
  | `CRITICAL` | `logging.critical()` | _(niveau le plus haut)_ erreur grave, ayant causé ou étant sur le point de causer l'arrêt du programme ou des conséquences majeures |

<br>
- C'est à vous de décider quel message tombe dans quelle catégorie !

- Si vous configurez le logging à un certain niveau (avec le paramètre `level` de `logging.basicConfig()`), seuls les messages de ce niveau et des niveaux plus élevés seront enregistrés :

  - par exemple, si vous configurez le niveau à `WARNING`, seuls les messages de niveau `WARNING`, `ERROR` et `CRITICAL` seront logués, tandis que les messages `DEBUG` et `INFO` seront ignorés

  - donc en phase de conception, on peut configurer le niveau à `DEBUG` pour tout voir et loguer sur la console, puis en production on peut le monter à `WARNING` ou `ERROR` pour ne loguer que les problèmes sérieux, tout en passant à un log sur fichier

#### Désactiver le logging

- Si vous souhaitez désactiver complètement le logging, il suffit d'appeler la fonction `logging.disable(logging.CRITICAL)` : cela indique au système de logging d'ignorer tous les messages de niveau `CRITICAL` et en dessous (ce qui revient à désactiver le logging)

  - si vous utilisez cette instruction, vous la placerez généralement au début du script, juste après l'import du module `logging` : il sera alors facile de la commenter/décommenter selon les besoins

---

### Autres outils de debugging

Si vous utilisez un environnement de développement intégré (IDE) comme VSCode, PyCharm, ou autre, vous disposez d'outils de debugging intégrés plus puissants pour analyser et corriger les bugs :

  - exécution pas à pas (_step-by-step execution_) : permet d'exécuter le code à votre rythme, ligne par ligne, pour observer comment les valeurs des variables évoluent

  - points d'arrêt (_breakpoints_) : permettent de suspendre l'exécution du programme à un certain endroit qui nous intéressent

  - inspection des variables : permet de voir les valeurs des variables dans le contexte d'exécution actuel

#### Exécution pas à pas sous VS Code

- Pour exécuter un script Python en mode debugging pas à pas (ligne par ligne) dans VS Code :

  - ouvrez le fichier `.py` dans l'éditeur

  - ouvrez le panneau « _Run and Debug_ » sur la gauche

  - placez un point d'arrêt (_breakpoint_) en début de script en cliquant dans la marge gauche de la ligne (indication « _click to add a breakpoint_ ») : un point rouge doit apparaître dans la marge

  - cliquez sur le bouton _`Run and Debug`_

  - le script démarre mais s'arrête immédiatement au point d'arrêt : il est « en pause »

  - vous pouvez maintenant contrôler l'exécution pas à pas avec les boutons en haut du panneau (ou avec les raccourcis clavier) :

    - _Step Over (F10)_ : exécute la ligne courante et s'arrête de nouveau à la suivante

    - _Step Into (F11)_ : idem, mais si la ligne courante contient un appel de fonction, on entre dans la fonction et s'arrête à la première ligne de celle-ci pour continuer le debugging à l'intérieur

    - _Step Out (Shift+F11)_ : termine l'exécution de la fonction courante et revient à l'appelant

  - dans le panneau de gauche, vous pouvez voir les variables locales et globales, et leurs valeurs actuelles ; vous pouvez aussi survoler les variables dans le code pour voir leur valeur

- Pour déboguer pas à pas, utiliser _Step Over_ est beaucoup plus fréquent que _Step Into_ : on n'utilise _Step Into_ que lorsqu'on soupçonne qu'une fonction appelée est la source du problème et qu'on veut vraiment « sauter » dedans pour voir ce qui s'y passe

- Parfois, on ne souhaite pas exécuter le script depuis le début, mais directement à un certain endroit où l'on soupçonne que quelque chose ne fonctionne pas bien : dans ce cas, on place le point d'arrêt à l'endroit souhaité avant de lancer le debugging

- Il est aussi possible de placer plusieurs points d'arrêt dans le script : l'exécution s'arrêtera dès qu'ils seront rencontrés dans le flux d'exécution

- Quand vous avez vu ce que vous aviez à voir et que vous voulez continuer le script normalement, cliquez sur le bouton _Continue (F5)_ : le script reprend son exécution normale jusqu'à la fin (ou jusqu'au prochain point d'arrêt rencontré)

- Si vous souhaitez arrêter immédiatement l'exécution du script, cliquez sur le bouton _Stop_ (carrée rouge)

---

### Asserting et Logging - Exemple

- Éventuellement, copiez le script suivant dans un fichier `.py` et ouvrez-le sous VS Code

- Le script récupère les notes entrées par l'utilisateur, puis calcule et affiche la moyenne

- Lorsqu'on l'exécute, on constate un bug : la moyenne affichée est incorrecte

- Ajoutez le support du logging sur ce script pour diagnostiquer le problème

  - insérer des logs de niveaux pertinents : ici on aura du `DEBUG` pour suivre les valeurs des variables de manière détaillé (et voir à quel moment surgit le problème rencontré), et du `INFO` pour indiquer où on en est dans le traitement global (suggestion : début/fin de la saisie des notes, saisie d'une note (?), calcul de la moyenne)

- Ajoutez des assertions pour vérifier certains invariants dans le code

  - les notes entrées doivent être comprises entre 0 et 20
  
  - dans la fonction `calculate_average()`, le nombre de notes doit être supérieur à zéro avant de faire la division

In [ ]:
def calculate_average(grade_sum, number_of_grades):
  grade_average = int(grade_sum / number_of_grades)
  return grade_average

print()
counter = 0
total = 0
while True:
  grade = input('Entrez une note ("fini" pour terminer) : ')
  if grade == 'fini':
    break
  counter = counter + 1
  total = total + int(grade)

average = calculate_average(counter, total)
print('Moyenne : ', average)

---

### Debugging - Exemple

- Copiez le script suivant dans un fichier `.py` et ouvrez-le sous VS Code

- Le script demande des années à l'utilisateur et indique si elles sont bissextiles ou non

- Il y a un bug : `2100` est une année non bissextile, mais le script l'indique comme bissextile

- Déboguez le script en utilisant les outils de debugging de VS Code

  - où allez-vous placer le ou les points d'arrêt ?

  - quelles variables allez-vous inspecter à quel moment ?

In [14]:
def is_leap_year(year):
  if year % 4 == 0:
    if year % 100 == 0:
      if year % 400 == 0:
        return True
      return True
    return True
  return False

while True:
  year = input('Entrez une année ("fini" pour terminer) : ')
  if year == 'fini':
    break
  if (is_leap_year(int(year))):
    print(year + ' est bissextile.')
  else:
    print(year + ' n\'est pas bissextile.')

print('FIN')

1995 n'est pas bissextile.
1996 est bissextile.
2000 est bissextile.
2020 est bissextile.
2100 est bissextile.
2400 est bissextile.
FIN


---

### Synthèse

- **Assertions** : vérification d'hypothèses sur le code ; à utiliser pour **vérifier que les données sont conformes aux attentes** avant de les traiter ; seulement pour les situations 

- **Tracer une exécution avec `print()`** : technique simple mais limitée ; **à n'utiliser que pour de petits scripts** ou scripts ***one-shot***

- **Journalisation (Logging)** : technique robuste et flexible pour enregistrer des messages d'information sur l'exécution d'un programme ; à utiliser pour les scripts en production ; savoir **configurer les niveaux de log** et les **canaux de sortie** ; comprendre, selon le contexte, **quels messages loguer** et **à quel niveau**

- **Debugging avec outils spécialisés** : exécution pas à pas, _breakpoints_, inspection des variables ; savoir localiser et corriger les bugs en inspectant _en live_ une exécution est une compétence importante

---